In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model


In [3]:
img_size = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    "animal_data",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="training"
)

val_data = datagen.flow_from_directory(
    "animal_data",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation"
)



Found 1574 images belonging to 16 classes.
Found 386 images belonging to 16 classes.


In [7]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # Freeze base model

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
output = Dense(16, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)


In [8]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [9]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)


Epoch 1/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.6830 - loss: 1.1416 - val_accuracy: 0.8446 - val_loss: 0.5423
Epoch 2/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 34s 680ms/step - accuracy: 0.9250 - loss: 0.2908 - val_accuracy: 0.8601 - val_loss: 0.4553
Epoch 3/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 34s 680ms/step - accuracy: 0.9600 - loss: 0.1598 - val_accuracy: 0.9016 - val_loss: 0.4017
Epoch 4/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 34s 673ms/step - accuracy: 0.9771 - loss: 0.1034 - val_accuracy: 0.8834 - val_loss: 0.3570
Epoch 5/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 34s 679ms/step - accuracy: 0.9860 - loss: 0.0706 - val_accuracy: 0.9041 - val_loss: 0.3688


In [11]:
model.save("animal_model.h5")